In [16]:
from google.colab import files
upload = files.upload()

Saving train.csv to train (1).csv


Run this code in a code cell before answering any of the question. This will create your knowledge base and the FAISS index for this milestone.

In [17]:
!pip install faiss-cpu

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('train.csv')

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base successfully created


Zero-shot classifier for Q1, Q2, Q6

In [18]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Question 1 - Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)?

In [19]:
result = zs(prompt_150, candidate_labels=labels_150)

correct_score = dict(zip(result["labels"], result["scores"]))[ans_150]

print("Ground Truth Answer:", ans_150)
print("Predicted Probability:", correct_score)

Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Predicted Probability: 0.3844209909439087


Question 2 - Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [20]:
query_embedding = model.encode([prompt_150])

k = 10
distances, indices = index.search(query_embedding, k)

print("Top-10 Retrieved KB Indices:", indices[0])

if 150 in indices[0]:
    rank = np.where(indices[0] == 150)[0][0] + 1
    print("Rank of the true document:", rank)
else:
    print("True document not found in the top 10.")

Top-10 Retrieved KB Indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of the true document: 10


Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)

In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder.
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

Code to use a Cross-Encoder

In [21]:
retrieved_indices = indices[0]
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Question 3 - Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [22]:
sorted_order = np.argsort(ce_scores)[::-1]
reranked_indices = retrieved_indices[sorted_order]

print("Reranked KB Indices:", reranked_indices)

if 150 in reranked_indices:
    rank = np.where(reranked_indices == 150)[0][0] + 1
    print("Rank of the true document after Cross-Encoder reranking:", rank)
else:
    print("True document not found in the reranked top 10.")

Reranked KB Indices: [ 150 1906  847 1693 1269 1532  168  576 1701  663]
Rank of the true document after Cross-Encoder reranking: 1


Question 4 - Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [23]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

query_embedding = model.encode([prompt_42])

k = 5
distances, indices = index.search(query_embedding, k)

retrieved_docs = [kb[i] for i in indices[0]]
concatenated_docs = " ".join(retrieved_docs)

input_text = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer(input_text, truncation=False)

num_tokens = len(tokens["input_ids"])
print("Total number of tokens:", num_tokens)

Total number of tokens: 216


Question 5 - Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option?

In [24]:
true_document = kb[150]

rag_input = f"Context: {true_document} Question: {prompt_150}"

rag_result = zs(rag_input, candidate_labels=labels_150)

correct_score = dict(zip(rag_result["labels"], rag_result["scores"]))[ans_150]

print("Ground Truth Answer:", ans_150)
print("Predicted Probability:", correct_score)

Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Predicted Probability: 0.9894258379936218


Concept: The Danger of Bad Retrieval (Adversarial RAG)

The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

Question 6 - What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now?

In [25]:
wrong_document = kb[999]

adversarial_input = f"Context: {wrong_document} Question: {prompt_150}"

adv_result = zs(adversarial_input, candidate_labels=labels_150)

correct_score = dict(zip(adv_result["labels"], adv_result["scores"]))[ans_150]

print("Ground Truth Answer:", ans_150)
print("Probability of the correct option:", correct_score)

Ground Truth Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability of the correct option: 0.5289496779441833


In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Question 7 - For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [26]:
hits = 0
k = 5

for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])

    correct_text = str(row[row["answer"]])

    query_embedding = model.encode([prompt])
    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Hit Rate (%):", round(hit_rate, 1))

Hits: 73
Hit Rate (%): 73.0


Question 8 - Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [27]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

k = 5
map3_scores = []

for i in range(20):
    row = train.iloc[i]

    prompt = str(row["prompt"])
    correct_letter = row["answer"]
    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    query_embedding = model.encode([prompt])
    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    rag_input = f"Context: {best_doc} Question: {prompt}"

    result = zs(rag_input, candidate_labels=labels)

    text_to_letter = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }

    predicted_letters = [text_to_letter[label] for label in result["labels"]]
    top3 = predicted_letters[:3]

    if correct_letter in top3:
        score = 1 / (top3.index(correct_letter) + 1)
    else:
        score = 0

    map3_scores.append(score)

final_map3 = np.mean(map3_scores)

print("Average MAP@3:", round(final_map3, 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Average MAP@3: 0.975
